In [12]:
import requests
import pandas as pd
import time


In [2]:
url = "https://open.overheid.nl/overheid/openbaarmakingen/api/v0/zoek"
params = {
    "zoektekst": "klimaat",
    "start": 0,
    "aantalResultaten": 20
}

response = requests.get(url, params=params)
data = response.json()

display(data)


{'totaal': 112078,
 'resultaten': [{'document': {'id': 'woo-idx-7bdc240629fce05ad40bde680b9ec7c8ae09a9e0_1',
    'pid': 'https://open.overheid.nl/documenten/woo-idx-7bdc240629fce05ad40bde680b9ec7c8ae09a9e0',
    'titel': 'oplossingsrichtingen_voor_klimaatadaptatie_in_de_oosterschelde',
    'openbaarmakingsdatum': '2026-02-12',
    'weblocatie': 'https://open.rijkswaterstaat.nl/publish/pages/221537/oplossingsrichtingen_voor_klimaatadaptatie_in_de_oosterschelde.pdf',
    'publisher': 'Rijkswaterstaat',
    'aanbieder': 'Harvester',
    'mutatiedatumtijd': '2026-02-12T21:34:50.166Z'},
   'highlightedText': 'onderzoek naar de verbinding van de <b>klimaat</b>- en zandhongeropgaven met de economische gebruiksfuncties de ecologie en het landschap van de Oosterschelde Effecten Zeespiegelstijging en Zandhonger Oosterschelde EZZO fase 3. o Monitoring en evaluatie ... om het beheer en beleid op te rekken voordat een knikpunt wordt bereikt Tabel 1-1 geeft een kwalitatief overzicht van • de relevan

In [3]:
print(data.keys())

dict_keys(['totaal', 'resultaten', 'filters'])


In [4]:
# voorbeeld metadata van document
data["resultaten"][0]


{'document': {'id': 'woo-idx-7bdc240629fce05ad40bde680b9ec7c8ae09a9e0_1',
  'pid': 'https://open.overheid.nl/documenten/woo-idx-7bdc240629fce05ad40bde680b9ec7c8ae09a9e0',
  'titel': 'oplossingsrichtingen_voor_klimaatadaptatie_in_de_oosterschelde',
  'openbaarmakingsdatum': '2026-02-12',
  'weblocatie': 'https://open.rijkswaterstaat.nl/publish/pages/221537/oplossingsrichtingen_voor_klimaatadaptatie_in_de_oosterschelde.pdf',
  'publisher': 'Rijkswaterstaat',
  'aanbieder': 'Harvester',
  'mutatiedatumtijd': '2026-02-12T21:34:50.166Z'},
 'highlightedText': 'onderzoek naar de verbinding van de <b>klimaat</b>- en zandhongeropgaven met de economische gebruiksfuncties de ecologie en het landschap van de Oosterschelde Effecten Zeespiegelstijging en Zandhonger Oosterschelde EZZO fase 3. o Monitoring en evaluatie ... om het beheer en beleid op te rekken voordat een knikpunt wordt bereikt Tabel 1-1 geeft een kwalitatief overzicht van • de relevante <b>klimaat</b> en socio-economische scenario-ele

In [5]:
doc = data["resultaten"][0]

doc_id = doc["document"]["id"]
titel = doc["document"]["titel"]
datum = doc["document"]["openbaarmakingsdatum"]
publisher = doc["document"]["publisher"]

print(doc_id)
print(titel)
print(datum)
print(publisher)


woo-idx-7bdc240629fce05ad40bde680b9ec7c8ae09a9e0_1
oplossingsrichtingen_voor_klimaatadaptatie_in_de_oosterschelde
2026-02-12
Rijkswaterstaat


In [10]:
def fetch_document_metadata(doc_id: str) -> dict:
    """
    Haalt uitgebreide metadata op voor één document via de juiste detail-endpoint.
    """
    # werkt aleen voor lijst van documenten
    # url = f"https://open.overheid.nl/overheid/openbaarmakingen/api/v0/zoek?zoektekst=&start=0&aantalResultaten=100/{doc_id}"

    url = f"https://open.overheid.nl/overheid/openbaarmakingen/api/v0/zoek/{doc_id}"   

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()
    doc = data.get("document", {})

    # data verzamelen
    identifiers = doc.get("identifiers", [])
    publisher = doc.get("publisher", {}).get("label")
    titel = doc.get("titelcollectie", {}).get("officieleTitel")

    informatiecategorieen = [
        cat.get("label")
        for cat in doc.get("classificatiecollectie", {}).get("informatiecategorieen", [])
    ]

    # PDF download URL
    pdf_url = None
    versies = data.get("versies", [])
    if versies and versies[0].get("bestanden"):
        pdf_url = versies[0]["bestanden"][0].get("id")


    return {
        "doc_id": doc_id,
        "identifiers": identifiers,
        "publisher": publisher,
        "titel": titel,
        "informatiecategorieen": informatiecategorieen,
        "pdf_url": pdf_url
    }


In [11]:
# ff testen
doc_id = data["resultaten"][0]["document"]["id"]
metadata = fetch_document_metadata(doc_id)
metadata


{'doc_id': 'woo-idx-7bdc240629fce05ad40bde680b9ec7c8ae09a9e0_1',
 'identifiers': [],
 'publisher': 'Rijkswaterstaat',
 'titel': 'oplossingsrichtingen_voor_klimaatadaptatie_in_de_oosterschelde',
 'informatiecategorieen': ['onderzoeksrapporten'],
 'pdf_url': None}

In [ ]:
def fetch_multiple_documents(target_n=100, page_size=20):
    """
    Haalt meerdere documenten op via de zoek-API,
    roept voor elk document de detail-endpoint aan,
    en retourneert een lijst met metadata (alleen met PDF).
    """

    base_url = "https://open.overheid.nl/overheid/openbaarmakingen/api/v0/zoek"
    
    all_metadata = []
    start = 0

    while len(all_metadata) < target_n:
        
        # Zoek-pagina ophalen
        params = {
            "zoektekst": "",
            "start": start,
            "aantalResultaten": page_size
        }

        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        results = data.get("resultaten", [])

        if not results:
            break  # Stop als er geen resultaten meer zijn

        # Loop over resultaten van deze pagina
        for result in results:
            doc_id = result["document"]["id"]

            metadata = fetch_document_metadata(doc_id)

            # Alleen toevoegen als er een PDF is
            if metadata["pdf_url"] is not None:
                all_metadata.append(metadata)

            # Stop zodra we genoeg hebben
            if len(all_metadata) >= target_n:
                break

            time.sleep(0.2)  # kleine pauze om API niet te overbelasten

        start += page_size  # Ga naar volgende pagina

    return all_metadata


In [20]:
# Lijst om alle metadata in op te slaan
all_metadata = []

# Loop over de 20 zoekresultaten
for result in data["resultaten"]:
    doc_id = result["document"]["id"]
    
    # Detail-metadata ophalen
    metadata = fetch_document_metadata(doc_id)
    
    all_metadata.append(metadata)

len(all_metadata)


20

In [21]:
# Zet de lijst met dicts om naar een DataFrame
df = pd.DataFrame(all_metadata)

# Bekijk de eerste paar rijen
df.head()

,doc_id,identifiers,publisher,titel,informatiecategorieen
0,oep-48ca6050a4cfde1d79a21d5c7a8c9c9ea2994e3e_3,[kst-31793-290],Onbekend,"31793, nr. 290 - Internationale klimaatafspraken",[vergaderstukken Staten-Generaal]
1,oep-53462895e5fbb95441ee3ace5bff2377d6665bda_1,[kst-31793-293],Onbekend,"31793, nr. 293 - Internationale klimaatafspraken",[vergaderstukken Staten-Generaal]
2,oep-86fe86fac024ed514c0e3e8ccb53b4d6bee44bff_1,[kst-31793-295],Onbekend,"31793, nr. 295 - Internationale klimaatafspraken",[vergaderstukken Staten-Generaal]
3,oep-b3c6570e78da3cfa1a904799b85fdf6023da84a9_1,[kst-31793-292],Onbekend,"31793, nr. 292 - Internationale klimaatafspraken",[vergaderstukken Staten-Generaal]
4,oep-b8c3a61d94ca991572a9b953b326d3daafe9d6e8_1,[kst-31793-291],Onbekend,"31793, nr. 291 - Internationale klimaatafspraken",[vergaderstukken Staten-Generaal]


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   doc_id                 20 non-null     str   
 1   identifiers            20 non-null     object
 2   publisher              20 non-null     str   
 3   titel                  20 non-null     str   
 4   informatiecategorieen  20 non-null     object
dtypes: object(2), str(3)
memory usage: 932.0+ bytes


In [ ]:
# # optioneel de test-set opslaan als csv
# df.to_csv("ground_truth_20_docs.csv", index=False)